<a href="https://colab.research.google.com/github/AzzamAlnatsheh/DDS_BuildingAIChallenge/blob/main/Dox_using_Agno_Final_Version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dox the Cheat Sheet AI Advisor

Installing the necessary packages

In [ ]:
%%capture --no-display
!pip install agno gradio duckduckgo-search python-dotenv lancedb  pypdf tantivy
!pip install -U ddgs
!pip install pypdf
!pip install pylance
!pip install --upgrade gradio
!pip install PyMuPDF gradio

In [ ]:
import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [ ]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('OPENAI_NEW_API_KEY')

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.knowledge.embedder.openai import OpenAIEmbedder #way of importing the embedder has changed
from agno.tools.duckduckgo import DuckDuckGoTools
# from agno.knowledge.pdf_url import PDFUrlKnowledgeBase
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.reader.pdf_reader import PDFReader
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.tools.python import PythonTools

# Load the key securely
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_NEW_API_KEY")

# Create a knowledge base with PDF documents
knowledge = Knowledge(
        vector_db=LanceDb(
            uri="tmp/lancedb",
            table_name="pdf_documents",
            search_type=SearchType.hybrid,
            embedder=OpenAIEmbedder(id="text-embedding-3-small"),
        )
)

import requests

pdf_urls = ["https://media.datacamp.com/cms/working-with-hugging-face.pdf",
            "https://media.datacamp.com/cms/ai-agents-cheat-sheet.pdf",
            "https://media.datacamp.com/cms/introduction-to-sql-with-ai-1.pdf",
            "https://media.datacamp.com/legacy/image/upload/v1719844709/Marketing/Blog/Azure_CLI_Cheat_Sheet.pdf"]

def download_if_needed(url, filename):
    if not os.path.exists(filename):
        response = requests.get(url)
        with open(filename, "wb") as f:
            f.write(response.content)

for i, url in enumerate(pdf_urls):
    filename = f"file_{i}.pdf"

    download_if_needed(url, filename)

    knowledge.insert(
        path=filename,
        reader=PDFReader(),
        metadata={"source": url}
    )

print("All PDFs added successfully ✅")

# Define the agent
agent = Agent(
    model=OpenAIChat(id="gpt-4.1-mini", temperature = 0.2),
    description="You are Dox a data expert!",
    instructions="""
    You are a data professional's assistant named Dox.

    Your primary goal is to answer data-related questions accurately and concisely, citing all sources.

    Here are your operating procedures:

    1.  **Question Relevance Check**:
        *   Always start by checking if the question is data-related.
        *   If the question is NOT data-related, you MUST respond with: "Please ask relevant data questions only." and terminate.

    2.  **Information Gathering Strategy**:
        *   **Prioritize Knowledge Base**: First, search your internal knowledge base for the answer.
        *   **Supplement with Web Search**: If the knowledge base information is outdated, insufficient, or the question is better suited for current web information, use the DuckDuckGo tool to perform web searches to fill in gaps or find the most up-to-date data.

    3.  **Response Length Guidelines**:
        *   For basic questions, keep your answer to a maximum of 300 words.
        *   For complex questions, extend your answer to a maximum of 500 words.

    4.  **Citation Rules (CRITICAL)**:
        *   **Knowledge Base Citation**: For any information sourced from your internal knowledge base, you MUST include a citation on a NEW LINE after the answer, starting with "Source: ", followed by the metadata field 'source' to get the hyperlink.
        *   **Web Search Citation**: For any information obtained from the web using the DuckDuckGo tool, you MUST include a citation on a NEW LINE after the answer, starting with "Online Source: ", followed by the full hyperlink.
        *   **Final Rule for Citations**: Always end your answers with the appropriate citations, ensuring they are on separate lines as specified. Do NOT mix or combine citation types on a single line.
        * ALWAYS cite with links NOT text like "from internal knowledge base"

    5.  **Accuracy and Non-Hallucination**:
        *   Provide factual and relevant answers based ONLY on the information found in your knowledge base or through web searches.
        *   NEVER invent or hallucinate information. If an answer cannot be found, state that directly.

    Make sure to follow these instructions precisely.
    """,
    knowledge=knowledge,
    add_datetime_to_context=True,
    add_location_to_context=True,
    search_knowledge=True,
    tools=[DuckDuckGoTools(), PythonTools()],
    markdown=True
)

INFO Adding content from path, a7799010-8bf0-5574-80d2-444bb3aff135, None, file_0.pdf, None

INFO Deleted 2 records with content_hash '8b9b4a53cdb641de63637e6820105dd162372f3b2467f3327213bff1fa12c8e8' from   
     table 'pdf_documents'.

INFO Adding content from path, 3f5658ca-8a87-560d-b7c3-e5eb701721de, None, file_1.pdf, None

INFO Deleted 3 records with content_hash '37a0f5359625f5a12411149393ed27ef5757e110b35de5e7cbb8ba4810e676aa' from   
     table 'pdf_documents'.

INFO Adding content from path, fae0f5a0-ccc1-587b-acd4-697b3d418c34, None, file_2.pdf, None

INFO Deleted 4 records with content_hash 'c7fbd9f80d1adf0e34a7287f433892a6b2d45146a84bfaff217dd97be1e07cfb' from   
     table 'pdf_documents'.

INFO Adding content from path, ebdc40e9-1695-5b40-b040-33d59c97543d, None, file_3.pdf, None

INFO Deleted 3 records with content_hash '30cc18cf5fb2513091e2cd2074844e6c346ceeb8e589fd7425a7276c4b1f1016' from   
     table 'pdf_documents'.

All PDFs added successfully ✅


In [ ]:
!pip show gradio

Name: gradio
Version: 6.12.0
Summary: Python library for easily interacting with trained machine learning models
Home-page: https://github.com/gradio-app/gradio
Author: 
Author-email: Abubakar Abid <gradio-team@huggingface.co>, Ali Abid <gradio-team@huggingface.co>, Ali Abdalla <gradio-team@huggingface.co>, Dawood Khan <gradio-team@huggingface.co>, Ahsen Khaliq <gradio-team@huggingface.co>, Pete Allen <gradio-team@huggingface.co>, Ömer Faruk Özdemir <gradio-team@huggingface.co>, Freddy A Boulton <gradio-team@huggingface.co>, Hannah Blair <gradio-team@huggingface.co>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, brotli, fastapi, gradio-client, groovy, hf-gradio, httpx, huggingface-hub, jinja2, markupsafe, numpy, orjson, packaging, pandas, pillow, pydantic, pydub, python-multipart, pytz, pyyaml, safehttpx, semantic-version, starlette, tomlkit, typer, typing-extensions, uvicorn
Required-by: 


In [ ]:
import gradio as gr
import fitz  # PyMuPDF
from PIL import Image
import io
import requests
import re

# -----------------------------
# Your agent function
# -----------------------------
def ask_agent(question):
    response = agent.run(question, use_knowledge=True)
    full_content = response.get_content_as_string()

    match = re.search(r'https?://[^\s]+\.pdf', full_content, re.IGNORECASE)
    link = match.group(0) if match else None

    return full_content, link

# -----------------------------
# Download PDF
# -----------------------------
def download_pdf_from_url(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.content

# -----------------------------
# Display PDF
# -----------------------------
def display_pdf(pdf_url):
    if not pdf_url:
        return gr.update(visible=False)

    try:
        pdf_bytes = download_pdf_from_url(pdf_url)
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        page = doc[0]

        zoom = 1.5
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)

        img_data = pix.tobytes("png")
        img = Image.open(io.BytesIO(img_data))
        doc.close()

        return gr.update(value=img, visible=True)
    except Exception as e:
        print(f"Error: {str(e)}")
        return None

# -----------------------------
# UI Wrapper for Blocks
# -----------------------------
def ask_agent_ui(question):
    response_text, link = ask_agent(question)

    return (
        response_text,
        link,
        gr.update(visible=link is not None),  # button visibility
        gr.update(value=None, visible=False)  # RESET PDF preview
    )

# -----------------------------
# Combined Gradio UI with Blocks and Interface
# -----------------------------
with gr.Blocks(title="AI + PDF Viewer", theme=gr.themes.Ocean()) as demo:
    gr.Markdown("# 🌊 Dox the Data Professional's Advisor 🤖")
    gr.Markdown("🧠 Dox has 4 DataCamp cheat sheets that you could ask it about (AI Agents | Azure CLI | Hugging Face | SQL with AI):")

    # Create two columns for better layout
    with gr.Row():
        with gr.Column(scale=2):
            # Embed the Interface components manually
            question = gr.Textbox(
                label="Ask Dox a question:",
                lines=2,
                placeholder="Type your question here..."
            )

            # Add examples
            gr.Examples(
                examples=[
                    "How do you log into Azure using device code authentication?",
                    "What are the three main components of an AI agent?",
                    "What are the “core four” Hugging Face libraries?",
                    "What SQL clause is used to filter data after grouping?"#,
                    # "How to do text summarizations?",
                    # "What is the Hugging Face Hub?",
                    # "Compare AI agents and traditional software systems.",
                    # "Where is the closest bakery?",
                    # "What is the latest OpenAI model?",
                    # "Give an example of an AI agent that uses a database."
                ],
                inputs=question,
                label="Example Questions"
            )

            ask_btn = gr.Button("Submit", variant="primary")

            # Output with copy button
            answer = gr.Markdown(
                label="Answer: ",
                render=True,
                container=True,
                elem_id="answer_markdown"
            )

        with gr.Column(scale=1):
            link_state = gr.State()
            show_btn = gr.Button("Show PDF", visible=False, variant="secondary")
            output_image = gr.Image(label="PDF Preview (Page 1)", visible=False)

    # Ask agent functionality
    ask_btn.click(
        ask_agent_ui,
        inputs=question,
        outputs=[answer, link_state, show_btn, output_image]
    )

    # Show PDF functionality
    show_btn.click(
        display_pdf,
        inputs=link_state,
        outputs=output_image
    ).then(
        lambda: gr.update(visible=True),
        None,
        output_image
    )

demo.launch(share=True)

/tmp/ipykernel_3127/1877139071.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="AI + PDF Viewer", theme=gr.themes.Ocean()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b42aa5454aa40a4a84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
